# Predicting Corporate Bankruptcy from Financial Statements

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 3, Machine Learning Foundations

**Dataset:** Polish Companies Bankruptcy Data, UCI Machine Learning Repository (dataset 365), licence CC BY 4.0:
https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data

In this notebook I train a model that predicts from one year of financial
statement ratios whether a company will go bankrupt within the next three
years. The input is a set of 64 financial ratios computed from published
annual statements, the output is a bankruptcy risk score.

**Research question:** How well can supervised machine learning predict, from
one year of financial ratios, whether a company will go bankrupt within three
years, and how should the model scores be turned into decisions when
bankruptcies are rare?

I picked this problem because it continues my capstone series. In Project 1 I
built a cleaned dataset of Swiss legal entities from the GLEIF register, in
Project 2 I analyzed company mutations from the Swiss commercial gazette. A
later capstone project will build a due diligence agent that looks up a
company and assesses it. Such an agent needs exactly the decision modeled
here, given the numbers a company publishes, how urgently does a human
analyst need to look at it.

## Problem definition and dataset

**Task type:** supervised learning, binary classification. Given the financial ratios of one company in one year, the model predicts whether the company went bankrupt within the following three years.

The data was collected from the Emerging Markets Information Service (EMIS) and covers Polish companies, bankrupt ones observed in 2000 to 2012, still operating ones evaluated from 2007 to 2013. It was donated to the UCI Machine Learning Repository by Sebastian Tomczak and is described in Zieba, Tomczak and Tomczak (2016). Every label is a real observed company outcome, so the data is not synthetic. The archive contains five files, one per forecasting horizon, from `1year` (ratios from the first year, bankruptcy label after five years) to `5year` (ratios from the fifth year, label after one year). I use the `3year` file and leave the other four untouched, it is the largest of the five and its three-year horizon fits the due diligence use case, a risk screening wants warning ahead of time, not a confirmation shortly before the collapse. The bankruptcy share of under 5% makes this an imbalanced classification problem, which shapes the metric choice and the threshold analysis below.

I downloaded the archive on 2026-08-24 and keep it unchanged in the project folder. The notebook reads the file it needs directly from the archive, so no manual unpacking step is required.

## Setup

In [29]:
import io
import textwrap
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

ARCHIVE = "polish+companies+bankruptcy+data.zip"
YEARS = 3

## Load the dataset

I read the `3year` file directly from the downloaded archive and check that the size and the class counts match the published description.

In [30]:
def load_dataset(archive_path, years):
    """Load one horizon file of the UCI Polish bankruptcy archive.

    The archive contains five ARFF files, one per forecasting horizon,
    and years selects the file (1 to 5). An ARFF file is a CSV table
    with a small header of @attribute lines. The "?" placeholder
    becomes a proper missing value and the target column "bankrupt"
    comes back as integers, 1 means bankrupt.
    """
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(f"{years}year.arff") as file:
            text = io.TextIOWrapper(file, encoding="utf-8").read()
    n_attributes = text.lower().count("@attribute") - 1
    columns = [f"Attr{i}" for i in range(1, n_attributes + 1)] + ["bankrupt"]
    data_section = text[text.lower().index("@data"):].split("\n", 1)[1]
    df = pd.read_csv(io.StringIO(data_section), header=None,
                     names=columns, na_values=["?"])
    df["bankrupt"] = df["bankrupt"].astype(int)
    return df

In [31]:
df = load_dataset(ARCHIVE, YEARS)

bankrupt_count = int(df["bankrupt"].sum())
print(f"Companies: {df.shape[0]:,}, columns: {df.shape[1]} "
      f"(64 financial ratios + target)")
print(f"Bankrupt within {YEARS} years: {bankrupt_count:,} "
      f"({df['bankrupt'].mean():.2%})")

Companies: 10,503, columns: 65 (64 financial ratios + target)
Bankrupt within 3 years: 495 (4.71%)


The file matches the published description, 10,503 rows with 64 financial ratios each, of which 495 carry the bankrupt label (4.71%), so among roughly 21 companies only one goes bankrupt. The columns arrive as anonymous names `Attr1` to `Attr64`, so I keep the ratio definitions from the UCI page in a table and use it whenever a feature needs to be interpreted.

In [32]:
# ratio definitions from the UCI dataset page, index = column name
FEATURE_DESCRIPTIONS = {
    "Attr1": "net profit / total assets",
    "Attr2": "total liabilities / total assets",
    "Attr3": "working capital / total assets",
    "Attr4": "current assets / short-term liabilities",
    "Attr5": "[(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365",
    "Attr6": "retained earnings / total assets",
    "Attr7": "EBIT / total assets",
    "Attr8": "book value of equity / total liabilities",
    "Attr9": "sales / total assets",
    "Attr10": "equity / total assets",
    "Attr11": "(gross profit + extraordinary items + financial expenses) / total assets",
    "Attr12": "gross profit / short-term liabilities",
    "Attr13": "(gross profit + depreciation) / sales",
    "Attr14": "(gross profit + interest) / total assets",
    "Attr15": "(total liabilities * 365) / (gross profit + depreciation)",
    "Attr16": "(gross profit + depreciation) / total liabilities",
    "Attr17": "total assets / total liabilities",
    "Attr18": "gross profit / total assets",
    "Attr19": "gross profit / sales",
    "Attr20": "(inventory * 365) / sales",
    "Attr21": "sales (n) / sales (n-1)",
    "Attr22": "profit on operating activities / total assets",
    "Attr23": "net profit / sales",
    "Attr24": "gross profit (in 3 years) / total assets",
    "Attr25": "(equity - share capital) / total assets",
    "Attr26": "(net profit + depreciation) / total liabilities",
    "Attr27": "profit on operating activities / financial expenses",
    "Attr28": "working capital / fixed assets",
    "Attr29": "logarithm of total assets",
    "Attr30": "(total liabilities - cash) / sales",
    "Attr31": "(gross profit + interest) / sales",
    "Attr32": "(current liabilities * 365) / cost of products sold",
    "Attr33": "operating expenses / short-term liabilities",
    "Attr34": "operating expenses / total liabilities",
    "Attr35": "profit on sales / total assets",
    "Attr36": "total sales / total assets",
    "Attr37": "(current assets - inventories) / long-term liabilities",
    "Attr38": "constant capital / total assets",
    "Attr39": "profit on sales / sales",
    "Attr40": "(current assets - inventory - receivables) / short-term liabilities",
    "Attr41": "total liabilities / ((profit on operating activities + depreciation) * (12 / 365))",
    "Attr42": "profit on operating activities / sales",
    "Attr43": "rotation receivables + inventory turnover in days",
    "Attr44": "(receivables * 365) / sales",
    "Attr45": "net profit / inventory",
    "Attr46": "(current assets - inventory) / short-term liabilities",
    "Attr47": "(inventory * 365) / cost of products sold",
    "Attr48": "EBITDA (profit on operating activities - depreciation) / total assets",
    "Attr49": "EBITDA (profit on operating activities - depreciation) / sales",
    "Attr50": "current assets / total liabilities",
    "Attr51": "short-term liabilities / total assets",
    "Attr52": "(short-term liabilities * 365) / cost of products sold",
    "Attr53": "equity / fixed assets",
    "Attr54": "constant capital / fixed assets",
    "Attr55": "working capital",
    "Attr56": "(sales - cost of products sold) / sales",
    "Attr57": "(current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation)",
    "Attr58": "total costs / total sales",
    "Attr59": "long-term liabilities / equity",
    "Attr60": "sales / inventory",
    "Attr61": "sales / receivables",
    "Attr62": "(short-term liabilities * 365) / sales",
    "Attr63": "sales / short-term liabilities",
    "Attr64": "sales / fixed assets",
}


def describe_feature(name):
    """Return a readable label like "Attr1: net profit / total assets".

    Missing indicator columns produced by the imputer are mapped back
    to the ratio they belong to.
    """
    if name.startswith("missingindicator_"):
        base = name.removeprefix("missingindicator_")
        return f"{base} is missing ({FEATURE_DESCRIPTIONS[base]})"
    return f"{name}: {FEATURE_DESCRIPTIONS[name]}"


pd.DataFrame({"ratio definition": FEATURE_DESCRIPTIONS})

,ratio definition
Attr1,net profit / total assets
Attr2,total liabilities / total assets
Attr3,working capital / total assets
Attr4,current assets / short-term liabilities
Attr5,[(cash + short-term securities + receivables -...
...,...
Attr60,sales / inventory
Attr61,sales / receivables
Attr62,(short-term liabilities * 365) / sales
Attr63,sales / short-term liabilities


## Data checks

Before I model anything I want to see what a company row actually looks like, how complete the columns are, whether rows are duplicated and how the ratios are distributed.

In [33]:
df.head()

,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,bankrupt
0,0.174190,0.41299,0.14371,1.3480,-28.9820,0.60383,0.219460,1.1225,1.1961,0.46359,...,0.163960,0.375740,0.83604,0.000007,9.7145,6.2813,84.291,4.3303,4.0341,0
1,0.146240,0.46038,0.28230,1.6294,2.5952,0.00000,0.171850,1.1721,1.6018,0.53962,...,0.027516,0.271000,0.90108,0.000000,5.9882,4.1103,102.190,3.5716,5.9500,0
2,0.000595,0.22612,0.48839,3.1599,84.8740,0.19114,0.004572,2.9881,1.0077,0.67566,...,0.007639,0.000881,0.99236,0.000000,6.7742,3.7922,64.846,5.6287,4.4581,0
3,0.024526,0.43236,0.27546,1.7833,-10.1050,0.56944,0.024526,1.3057,1.0509,0.56453,...,0.048398,0.043445,0.95160,0.142980,4.2286,5.0528,98.783,3.6950,3.4844,0
4,0.188290,0.41504,0.34231,1.9279,-58.2740,0.00000,0.233580,1.4094,1.3393,0.58496,...,0.176480,0.321880,0.82635,0.073039,2.5912,7.0756,100.540,3.6303,4.6375,0


Each row is one company observation, the 64 ratio columns describe profitability, leverage, liquidity and turnover, and `bankrupt` is the target. The first five rows are all label 0, so the file likely groups the operating companies together and I must not rely on row order anywhere. The column scales are very different, the profitability ratios sit between minus one and one while `Attr5` or `Attr62` reach into the hundreds, so the logistic regression will need a scaler later.

In [34]:
print(f"Rows: {df.shape[0]:,}, columns: {df.shape[1]}")
df.info()

Rows: 10,503, columns: 65
<class 'pandas.DataFrame'>
RangeIndex: 10503 entries, 0 to 10502
Data columns (total 65 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Attr1     10503 non-null  float64
 1   Attr2     10503 non-null  float64
 2   Attr3     10503 non-null  float64
 3   Attr4     10485 non-null  float64
 4   Attr5     10478 non-null  float64
 5   Attr6     10503 non-null  float64
 6   Attr7     10503 non-null  float64
 7   Attr8     10489 non-null  float64
 8   Attr9     10500 non-null  float64
 9   Attr10    10503 non-null  float64
 10  Attr11    10503 non-null  float64
 11  Attr12    10485 non-null  float64
 12  Attr13    10460 non-null  float64
 13  Attr14    10503 non-null  float64
 14  Attr15    10495 non-null  float64
 15  Attr16    10489 non-null  float64
 16  Attr17    10489 non-null  float64
 17  Attr18    10503 non-null  float64
 18  Attr19    10460 non-null  float64
 19  Attr20    10460 non-null  float64
 20  Attr21    969

All 64 ratios arrive as float columns and the target as an integer column, so no type conversion is needed. The non-null counts differ from column to column, `Attr21` has 9,696 values, `Attr27` has 9,788, most others are complete or nearly complete. The "?" placeholder of the source file became proper missing values because I passed it as `na_values` at load time. I look at the missing values next.

In [35]:
missing_share = (df.isna().mean() * 100).sort_values(ascending=False)
print(f"Columns with missing values: {(missing_share > 0).sum()} of {df.shape[1]}")
print(f"Rows with at least one missing value: {df.isna().any(axis=1).mean():.1%}")
missing_share.head(10).round(2)

Columns with missing values: 44 of 65
Rows with at least one missing value: 53.5%


Attr37    45.09
Attr21     7.68
Attr27     6.81
Attr60     5.64
Attr45     5.63
Attr28     2.17
Attr53     2.17
Attr54     2.17
Attr64     2.17
Attr24     2.16
dtype: float64

The gaps are wider than the info output suggested, 44 of the 65 columns have missing values and 53.5% of the rows have at least one. The ratio definitions suggest that much of this missingness may be structural rather than accidental. `Attr37` (45.09% missing) divides by long-term liabilities, which is zero for a company without long-term debt. `Attr21` (7.68%) needs the previous year's sales, which a company observed for the first time does not have. `Attr27` (6.81%) divides by financial expenses, which can be zero. The dataset documentation does not state the cause of the gaps explicitly, so I treat this as a plausible reading, not as a fact. Either way a missing ratio may carry information about the company, and what to do with it is a decision for the preparation section.

In [36]:
duplicate_rows = df.duplicated()
in_duplicate_group = df.duplicated(keep=False)
feature_columns = list(df.columns[:-1])
label_conflicts = (df[df.duplicated(subset=feature_columns, keep=False)]
                   .groupby(feature_columns, dropna=False)["bankrupt"].nunique() > 1).sum()

print(f"Redundant duplicate rows (identical to an earlier row): {int(duplicate_rows.sum())}")
print(f"Rows that are part of a duplicate group: {int(in_duplicate_group.sum())}")
print(f"Feature-duplicate groups with conflicting labels: {int(label_conflicts)}")
print(f"Class of the redundant rows: "
      f"{df[duplicate_rows]['bankrupt'].value_counts().to_dict()}")

Redundant duplicate rows (identical to an earlier row): 87
Rows that are part of a duplicate group: 174
Feature-duplicate groups with conflicting labels: 0
Class of the redundant rows: {0: 85, 1: 2}


The file contains exact duplicates, 174 rows are part of a duplicate group and 87 of them are redundant copies of an earlier row, 85 operating and 2 bankrupt. No duplicate group carries conflicting labels, so the copies do not hurt the label quality, but they are a problem for the evaluation. A random split can put one copy in training and an identical copy in validation or test, and the model would then be tested partly on rows it has already seen. The preparation section therefore removes the redundant copies before any split.

In [37]:
df.drop(columns="bankrupt").describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Attr1,10503.0,0.053,0.648,-1.769200e+01,0.001,0.043,0.124,52.652
Attr2,10503.0,0.620,6.427,0.000000e+00,0.254,0.464,0.689,480.730
Attr3,10503.0,0.095,6.420,-4.797300e+02,0.017,0.199,0.420,17.708
Attr4,10485.0,9.980,523.692,2.000000e-03,1.040,1.606,2.960,53433.000
Attr5,10478.0,-1347.662,118580.569,-1.190300e+07,-52.071,1.579,56.084,685440.000
...,...,...,...,...,...,...,...,...
Attr60,9911.0,571.336,37159.672,0.000000e+00,5.533,9.952,20.936,3660200.000
Attr61,10486.0,13.935,83.704,-6.590000e+00,4.486,6.677,10.587,4470.400
Attr62,10460.0,135.537,25991.162,-2.336500e+06,40.737,70.664,118.220,1073500.000
Attr63,10485.0,9.095,31.419,-0.000000e+00,3.063,5.139,8.883,1974.500


The quartiles look like ordinary balance sheet arithmetic, the median company earns a net profit of 4.3% of total assets (`Attr1`) and finances 46.4% of its assets with debt (`Attr2`). The extremes do not. `Attr5` ranges from minus 11.9 million to 685,440 with a standard deviation over 100,000, and several other ratios reach similar magnitudes. These are not typos, they appear whenever the denominator of a ratio is close to zero, for example sales-based ratios for a company with almost no sales. I treat them as genuine but extreme values. The random forest is unaffected because tree splits only use the order of the values, while the logistic regression sees the ratios through a standard scaler and stays vulnerable to them, which is worth remembering when the two models are compared.

In [ ]:
class_counts = df["bankrupt"].value_counts()
print(class_counts)
print(f"Bankruptcy share: {df['bankrupt'].mean():.4%}")

495 of the 10,503 companies carry the bankrupt label, a share of 4.71%, so a model that never flags anything is already right about 95 of every 100 companies. That settles the metric question before a single model exists, accuracy cannot be the headline number here. Together with the 87 redundant duplicate rows, the widespread missing ratios and the extreme values from near-zero denominators, this is the list the preparation section has to answer.